In [11]:
processor = AutoProcessor.from_pretrained("qwen/Qwen2.5-32B")

In [ ]:
import os
import torch
import glob
from collections import OrderedDict
import re
import shutil
from pathlib import Path
from accelerate.utils import merge_fsdp_weights

ckpt_dir   = "/mnt/blob/ckpts_bugfix/DAPO-PPO/Qwen2.5-32B/global_step_340/actor"
base_model = "qwen/Qwen2.5-32B"
out_dir    = "/tmp/Qwen-2.5-32B"
os.makedirs(out_dir, exist_ok=True)

In [3]:
import os, re, torch
from collections import defaultdict
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer



# # -------- 1. load every rank checkpoint --------
regex   = re.compile(r"model_world_size_\d+_rank_(\d+)\.pt")
rank_sd = {}                              # rank → state-dict
for f in os.listdir(ckpt_dir):
    m = regex.match(f)
    if m:
        rank = int(m.group(1))
        rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")

world_size = len(rank_sd)
assert world_size > 0, "no rank files found"

# -------- 2. collect per-param slices --------
slices = defaultdict(list)                # param → [local_tensor per rank]
for rank in range(world_size):
    for k, v in rank_sd[rank].items():
        if isinstance(v, torch.distributed._tensor.DTensor):
            # grab the shard stored on this rank
            slices[k].append(v._local_tensor)     # private attr but works fine
        else:                                     # unsharded params / scalars
            slices[k] = [v] * world_size          # replicate so cat() is no-op

# -------- 3. reassemble full tensors --------
full = {}
for k, parts in slices.items():
    # if more than one shard, concatenate along dim 0
    full[k] = torch.cat(parts, dim=0) if len(parts) > 1 else parts[0]

# -------- 4. drop into an HF model and save --------
cfg   = AutoConfig.from_pretrained(base_model)
model = AutoModelForCausalLM.from_config(cfg)
model.load_state_dict(full, strict=False)
model.save_pretrained(out_dir)
AutoTokenizer.from_pretrained(base_model).save_pretrained(out_dir)

print("✅ merged checkpoint written to", out_dir)


/tmp/ipykernel_237078/1833113373.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")
Using the `SD

[2025-05-13 13:35:15,984] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


✅ merged checkpoint written to /tmp/Qwen-2.5-32B


In [17]:
import os
import sys
from contextlib import contextmanager
from copy import deepcopy
from dataclasses import dataclass, field
from enum import Enum
from functools import partial
from math import ceil
from pprint import pprint
from typing import Dict, Type
from uuid import uuid4
from collections import defaultdict

import numpy as np
import torch
from torch.utils.data import RandomSampler, SequentialSampler
from torchdata.stateful_dataloader import StatefulDataLoader
from transformers import AutoProcessor, AutoTokenizer

import hydra
import ray
from omegaconf import OmegaConf, open_dict
from tqdm import tqdm
from codetiming import Timer

# Extend import path and set env vars before project-local imports
sys.path.insert(0, "/home/aiscuser/verl")
os.environ["SGLANG_BLOCK_NONZERO_RANK_CHILDREN"] = "0"

# ─── verl project ──────────────────────────────────────────────────────────────
from verl import DataProto
from verl.third_party.sglang.entrypoint import CustomEngine

from verl.trainer.ppo.ray_trainer import RayPPOTrainer
from verl.trainer.main_ppo import (
    get_custom_reward_fn,
    get_running_jobs,
    log_using_devices,
    get_current_job_id,
    get_available_devices,
)
from verl.trainer.ppo import core_algos
from verl.trainer.ppo.metric_utils import (
    compute_data_metrics,
    compute_throughout_metrics,
    compute_timing_metrics,
    reduce_metrics,
    bootstrap_metric,
    calc_maj_val,
)

from verl.single_controller.base import Worker
from verl.single_controller.ray import (
    RayResourcePool,
    RayWorkerGroup,
    RayClassWithInitArgs,
)
from verl.single_controller.ray.base import create_colocated_worker_cls

from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from verl.protocol import (
    collate_fn as batch_collate_fn,
    pad_dataproto_to_divisor,
    unpad_dataproto,
)
from verl.utils.fs import copy_to_local
from verl.utils.seqlen_balancing import (
    get_seqlen_balanced_partitions,
    log_seqlen_unbalance,
)
from verl.utils.checkpoint.checkpoint_manager import find_latest_ckpt_path
from verl.utils.tracking import Tracking, ValidationGenerationsLogger
from verl.utils.model import compute_position_id_with_mask

import pandas as pd

from transformers import AutoTokenizer

from verl import DataProto
from verl.utils.fs import copy_to_local
from verl.workers.fsdp_workers import ActorRolloutRefWorker
from verl.utils.hdfs_io import makedirs
from verl.single_controller.ray import RayClassWithInitArgs, RayResourcePool, RayWorkerGroup


In [ ]:
os.environ["ENSURE_CUDA_VISIBLE_DEVICES"] = os.environ.get('CUDA_VISIBLE_DEVICES', '')
ray.init(runtime_env={
    'env_vars': {
        'TOKENIZERS_PARALLELISM': 'true',
        'NCCL_DEBUG': 'WARN',
        'VLLM_LOGGING_LEVEL': 'WARN',
        # 'RAY_EXPERIMENTAL_NOSET_ROCR_VISIBLE_DEVICES': '1',
    }
})


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Python version:,3.9.19
Ray version:,2.44.1
Dashboard:,http://127.0.0.1:8265


(ActorRolloutRefWorker pid=334212) os.environ['CUDA_VISIBLE_DEVICES']: 0
(ActorRolloutRefWorker pid=334212) os.environ['LOCAL_RANK']: 0
(ActorRolloutRefWorker pid=335703) os.environ['CUDA_VISIBLE_DEVICES']: 2
(ActorRolloutRefWorker pid=335703) os.environ['LOCAL_RANK']: 2
(ActorRolloutRefWorker pid=335704) os.environ['CUDA_VISIBLE_DEVICES']: 3
(ActorRolloutRefWorker pid=335704) os.environ['LOCAL_RANK']: 3
(ActorRolloutRefWorker pid=334212) Model config after override: Qwen2Config {
(ActorRolloutRefWorker pid=334212)   "_name_or_path": "qwen/Qwen2.5-32B",
(ActorRolloutRefWorker pid=334212)   "architectures": [
(ActorRolloutRefWorker pid=334212)     "Qwen2ForCausalLM"
(ActorRolloutRefWorker pid=334212)   ],
(ActorRolloutRefWorker pid=334212)   "attention_dropout": 0.0,
(ActorRolloutRefWorker pid=334212)   "eos_token_id": 151643,
(ActorRolloutRefWorker pid=334212)   "hidden_act": "silu",
(ActorRolloutRefWorker pid=334212)   "hidden_size": 5120,
(ActorRolloutRefWorker pid=334212)   "initial

In [ ]:
config = OmegaConf.load('/home/aiscuser/verl/verl/trainer/config/generation_dapo.yaml')
from pprint import pprint
from omegaconf import OmegaConf
pprint(OmegaConf.to_container(config, resolve=True))  # resolve=True will eval symbol values
OmegaConf.resolve(config)
local_path = copy_to_local(config.model.path)
from verl.utils import hf_tokenizer
tokenizer = hf_tokenizer(local_path)
processor = AutoProcessor.from_pretrained(local_path)

if config.rollout.temperature == 0.:
    assert config.data.n_samples == 1, 'When temperature=0, n_samples must be 1.'

# read dataset. Note that the dataset should directly contain chat template format (e.g., a list of dictionary)
dataset = pd.read_parquet(config.data.path)
chat_lst = dataset[config.data.prompt_key].tolist()

chat_lst = [chat.tolist() for chat in chat_lst]

tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ray_cls_with_init = RayClassWithInitArgs(cls=ray.remote(ActorRolloutRefWorker), config=config, role='rollout')
resource_pool = RayResourcePool(process_on_nodes=[config.trainer.n_gpus_per_node] * config.trainer.nnodes)
wg = RayWorkerGroup(resource_pool=resource_pool, ray_cls_with_init=ray_cls_with_init)
wg.init_model()


{'actor': {'fsdp_config': {'fsdp_size': -1},
           'strategy': 'fsdp',
           'ulysses_sequence_parallel_size': 1},
 'data': {'batch_size': 512,
          'n_samples': 1,
          'output_path': '/opt/tiger/math_Qwen2-7B-Instruct.parquet',
          'path': '~/data/gsm8k_eval.parquet',
          'prompt_key': 'prompt'},
 'model': {'external_lib': None, 'path': 'qwen/Qwen2.5-32B'},
 'rollout': {'disable_log_stats': True,
             'do_sample': True,
             'dtype': 'bfloat16',
             'enable_chunked_prefill': True,
             'enforce_eager': True,
             'free_cache_engine': True,
             'gpu_memory_utilization': 0.8,
             'group_shuffle': True,
             'ignore_eos': False,
             'load_format': 'dummy_dtensor',
             'log_prob_max_token_len_per_gpu': 10240,
             'log_prob_micro_batch_size': None,
             'log_prob_micro_batch_size_per_gpu': 8,
             'log_prob_use_dynamic_bsz': True,
             'max_

In [7]:
from verl.workers.reward_manager import DAPORewardManager
val_reward_fn = DAPORewardManager(tokenizer=tokenizer,
                                num_examine=0,
                                compute_score=None,
                                reward_fn_key='data_source')
def _default_compute_score(data_source, solution_str, ground_truth, extra_info=None):
    if data_source == 'openai/gsm8k':
        from . import gsm8k
        res = gsm8k.compute_score(solution_str, ground_truth)
    elif data_source in ['lighteval/MATH', 'DigitalLearningGmbH/MATH-lighteval']:
        from . import math
        res = math.compute_score(solution_str, ground_truth)
        # [Optional] Math-Verify Integration
        # For enhanced accuracy, consider utilizing Math-Verify (https://github.com/huggingface/Math-Verify).
        # Note: Math-Verify needs to be manually installed via pip: `pip install math-verify`.
        # To use it, override the `compute_score` function with the following implementation:

        # from . import math_verify
        # res = math_verify.compute_score(solution_str, ground_truth)
    elif data_source == 'dapo-math' or data_source.startswith("aime") or data_source == 'orz':
        from verl.utils.reward_score import math_dapo
        res = math_dapo.compute_score(solution_str, ground_truth)
    elif data_source in [
            'numina_aops_forum', 'numina_synthetic_math', 'numina_amc_aime', 'numina_synthetic_amc', 'numina_cn_k12',
            'numina_olympiads'
    ]:
        from . import prime_math
        res = prime_math.compute_score(solution_str, ground_truth)
    elif data_source in ['codecontests', 'apps', 'codeforces', 'taco']:
        from . import prime_code
        res = prime_code.compute_score(solution_str, ground_truth, continuous=True)
    elif data_source in ['hiyouga/geometry3k']:
        from . import geo3k
        res = geo3k.compute_score(solution_str, ground_truth)
    elif data_source in ['kk_logic']:
        from . import kk
        res = kk.compute_score(solution_str, ground_truth)
    else:
        raise NotImplementedError(f"Reward function is not implemented for {data_source=}")

    if isinstance(res, dict):
        return res
    elif isinstance(res, (int, float, bool)):
        return float(res)
    else:
        return float(res[0])
val_reward_fn.compute_score = _default_compute_score

# Ours

## GSM8K

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1
test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end


In [ ]:
correct = 0
for i in range(len(sample_scores)):
    if sample_scores[i] > 0:
        correct += 1
print(f"Ours GSM8K Accuracy: {correct / len(sample_scores)}")

Ours GSM8K Accuracy: 0.9393479909021987


## MATH 500

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4
test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours MATH500 Accuracy: 0.618


## Minerva

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Minerva Accuracy: 0.27205882352941174


## Olympiad

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6
test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Olympiad Accuracy: 0.3620178041543027


In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/amc_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [37]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([40, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([40, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline AMC23 Accuracy: 0.725


In [55]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/amc_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 40


In [56]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False

    test_batch = test_batch.repeat(repeat_times=32, interleave=True)

    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1280, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([1280, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline AMC23 Accuracy: 0.728125


In [45]:
data_source_lst = []
data_source_lst.append(test_batch.non_tensor_batch.get('data_source', ['unknown'] * reward_tensor.shape[0]))
data_sources = np.concatenate(data_source_lst, axis=0)

In [62]:
ray.shutdown()

## GSM8K

In [15]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1
test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end


In [36]:
correct = 0
for i in range(len(sample_scores)):
    if sample_scores[i] > 0:
        correct += 1
print(f"Ours GSM8K Accuracy: {correct / len(sample_scores)}")

Ours GSM8K Accuracy: 0.9393479909021987


## MATH 500

In [37]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [38]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4
test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours MATH500 Accuracy: 0.618


## Minerva

In [40]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [41]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Minerva Accuracy: 0.27205882352941174


## Olympiad

In [43]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [44]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6
test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Olympiad Accuracy: 0.3620178041543027


# Baseline

## GSM8K

In [10]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [11]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1
test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours GSM8K Accuracy: 0.9416224412433661


## MATH 500

In [12]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [13]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4
test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours MATH500 Accuracy: 0.626


## Minerva

In [14]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [15]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline Minerva Accuracy: 0.27941176470588236


## Olympiad

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6
test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Olympiad Accuracy: 0.3620178041543027


In [ ]:
ray.shutdown()

# Baseline

## GSM8K

In [20]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [21]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1
test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline GSM8K Accuracy: 0.9408642911296436


## MATH 500

In [23]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [24]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4
test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours MATH500 Accuracy: 0.622


## Minerva

In [25]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [26]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Minerva Accuracy: 0.2977941176470588


## Olympiad

In [27]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [28]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6
test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Olympiad Accuracy: 0.3664688427299703


## AMC23

In [32]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/amc_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 40


In [33]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([40, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([40, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline AMC23 Accuracy: 0.7


In [60]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/amc_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 40


In [61]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False

    test_batch = test_batch.repeat(repeat_times=32, interleave=True)

    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1280, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([1280, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline AMC23 Accuracy: 0.72109375


In [34]:
ray.shutdown()

# No Training

## GSM8K

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1
test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline GSM8K Accuracy: 0.9408642911296436


## MATH 500

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4
test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours MATH500 Accuracy: 0.622


## Minerva

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Minerva Accuracy: 0.2977941176470588


## Olympiad

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Ours Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6
test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Ours Olympiad Accuracy: 0.3664688427299703


## AMC23

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/amc_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 40


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([40, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([40, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline AMC23 Accuracy: 0.7


In [ ]:
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/amc_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 40


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False

    test_batch = test_batch.repeat(repeat_times=32, interleave=True)

    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Baseline AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1280, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0


test_output_gen_batch shape: torch.Size([1280, 16384])
test_output_gen_batch meta info: {}
validation generation end
Baseline AMC23 Accuracy: 0.72109375
